# Preprocessing Summary

## Objective

The objective of preprocessing was to convert the raw NASA battery MATLAB data into a clean, standardized, cycle-level time-series dataset that can be used by the other project members for feature engineering and machine learning.

## Data Used

Four Li-ion batteries were processed:

- B0005
- B0006
- B0007
- B0018

Only discharge operations were extracted for the primary processed dataset because each discharge operation contains a measured battery capacity, which is important for battery aging and SOH analysis.

## Preprocessing Steps

The following preprocessing steps were performed:

1. Loaded the raw MATLAB battery files from the NASA dataset.
2. Extracted the discharge operations from each battery.
3. Assigned a sequential cycle number to each discharge operation for each battery.
4. Extracted the relevant discharge measurements.
5. Converted the MATLAB structures into a tabular Pandas DataFrame.
6. Standardized the measurement names.
7. Checked for missing values.
8. Checked for duplicate records.
9. Checked data types.
10. Checked for invalid numerical values.
11. Validated cycle numbering and sample counts.
12. Saved the resulting standardized dataset as a CSV file.

## Standardized Dataset Structure

The final dataset contains the following columns:

| Column | Description |
|--------|-------------|
| `battery_id` | Identifier of the battery |
| `cycle` | Sequential discharge cycle number for that battery |
| `time` | Time elapsed during the discharge operation (seconds) |
| `voltage` | Measured battery terminal voltage (Volts) |
| `current` | Measured battery current (Amps) |
| `temperature` | Measured battery temperature (°C) |
| `current_load` | Current measured at the load (Amps) |
| `voltage_load` | Voltage measured at the load (Volts) |
| `capacity` | Measured discharge capacity (Ah) |

## Final Dataset Statistics

| Battery | Discharge Cycles | Minimum Samples/Cycle | Maximum Samples/Cycle | Average Samples/Cycle |
|---------|-----------------:|-----------------------:|-----------------------:|----------------------:|
| B0005 | 168 | 179 | 371 | 299.32 |
| B0006 | 168 | 179 | 371 | 299.32 |
| B0007 | 168 | 179 | 371 | 299.32 |
| B0018 | 132 | 200 | 366 | 264.14 |
| **Total** | **636** | | | |

The final dataset contains **185,721 measurement records** across the 636 discharge cycles.

## Data Quality Validation

The processed dataset was validated after saving.

Results:

- Missing values: **0**
- Duplicate rows: **0**
- Invalid cycle numbers: **0**
- Negative time values: **0**
- Non-positive voltage values: **0**
- Non-positive capacity values: **0**
- All four expected battery IDs are present.
- All expected discharge cycles are present.
- Each discharge cycle contains exactly one capacity value.

## Variable-Length Time Series

The number of measurements differs between discharge cycles. Therefore, the processed dataset preserves the original variable-length time-series structure rather than artificially padding, truncating, or resampling the measurements.

Any further resampling, feature extraction, or transformation required for machine learning will be performed during the feature-engineering stage.

## Missing Values in Raw Data

During dataset understanding, six NaN values were identified in the charge measurements of B0018. Since the current standardized dataset contains only discharge operations, these values are not present in the processed discharge dataset.

No raw data was modified or deleted.

## Output File

The standardized processed dataset was saved at:

`data/processed/battery_discharge_cycles.csv`

The raw dataset under `data/raw/` remains unchanged and is treated as read-only.

## SOH Definition

State of Health (SOH) has not been added as a derived column at this stage. The final SOH definition and target formulation will be decided in consultation with the project team before feature engineering and machine learning.

## Preprocessing Status

The standardized discharge dataset has been successfully created and validated and is ready to be used as the input for the next stages of the BatteryIQ project.

In [1]:
import numpy as np
import pandas as pd
from scipy.io import loadmat
from pathlib import Path

print("Preprocessing environment ready!")

Preprocessing environment ready!


In [2]:
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

# Data directories
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Create processed directory if needed
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:")
print(PROJECT_ROOT)

print("\nRaw data directory:")
print(RAW_DIR)

print("\nProcessed data directory:")
print(PROCESSED_DIR)

print("\nRaw directory exists:", RAW_DIR.exists())
print("Processed directory exists:", PROCESSED_DIR.exists())

Project root:
d:\VIT Academic\TY\sem 5\EDI\BatteryIQ

Raw data directory:
d:\VIT Academic\TY\sem 5\EDI\BatteryIQ\data\raw

Processed data directory:
d:\VIT Academic\TY\sem 5\EDI\BatteryIQ\data\processed

Raw directory exists: True
Processed directory exists: True


In [3]:
battery_data = {}

battery_ids = ["B0005", "B0006", "B0007", "B0018"]

for battery_id in battery_ids:
    mat_path = RAW_DIR / f"{battery_id}.mat"
    
    battery_data[battery_id] = loadmat(mat_path)

print("Loaded batteries:")

for battery_id in battery_data:
    print(f"{battery_id}: loaded successfully")

Loaded batteries:
B0005: loaded successfully
B0006: loaded successfully
B0007: loaded successfully
B0018: loaded successfully


In [4]:
discharge_records = []
cycle_counts = {}

for battery_id, battery in battery_data.items():

    cycle_data = battery[battery_id]["cycle"][0, 0]

    discharge_index = 0

    for i in range(cycle_data.shape[1]):

        operation = cycle_data[0, i]

        if operation["type"].item() != "discharge":
            continue

        discharge_index += 1

        data = operation["data"][0, 0]

        time = np.asarray(data["Time"]).flatten()
        voltage = np.asarray(data["Voltage_measured"]).flatten()
        current = np.asarray(data["Current_measured"]).flatten()
        temperature = np.asarray(data["Temperature_measured"]).flatten()
        current_load = np.asarray(data["Current_load"]).flatten()
        voltage_load = np.asarray(data["Voltage_load"]).flatten()

        capacity = float(data["Capacity"].flatten()[0])

        for j in range(len(time)):
            discharge_records.append({
                "battery_id": battery_id,
                "cycle": discharge_index,
                "time": time[j],
                "voltage": voltage[j],
                "current": current[j],
                "temperature": temperature[j],
                "current_load": current_load[j],
                "voltage_load": voltage_load[j],
                "capacity": capacity
            })

    cycle_counts[battery_id] = discharge_index

print("Discharge cycles extracted:")
for battery_id, count in cycle_counts.items():
    print(f"{battery_id}: {count}")

print("\nTotal discharge cycles:", sum(cycle_counts.values()))
print("Total measurement records:", len(discharge_records))

Discharge cycles extracted:
B0005: 168
B0006: 168
B0007: 168
B0018: 132

Total discharge cycles: 636
Total measurement records: 185721


In [5]:
df = pd.DataFrame(discharge_records)
df = df.sort_values(
    ["battery_id", "cycle", "time"]
).reset_index(drop=True)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())

Dataset shape: (185721, 9)

Columns:
['battery_id', 'cycle', 'time', 'voltage', 'current', 'temperature', 'current_load', 'voltage_load', 'capacity']

First 5 records:


,battery_id,cycle,time,voltage,current,temperature,current_load,voltage_load,capacity
0,B0005,1,0.000,4.191492,-0.004902,24.330034,-0.0006,0.000,1.856487
1,B0005,1,16.781,4.190749,-0.001478,24.325993,-0.0006,4.206,1.856487
2,B0005,1,35.703,3.974871,-2.012528,24.389085,-1.9982,3.062,1.856487
3,B0005,1,53.781,3.951717,-2.013979,24.544752,-1.9982,3.030,1.856487
4,B0005,1,71.922,3.934352,-2.011144,24.731385,-1.9982,3.011,1.856487


In [6]:
print("=== MISSING VALUES ===")
print(df.isna().sum())

print("\n=== DUPLICATE RECORDS ===")
print("Duplicate rows:", df.duplicated().sum())

print("\n=== DATA TYPES ===")
print(df.dtypes)

print("\n=== NUMERICAL SUMMARY ===")
display(df.describe())

=== MISSING VALUES ===
battery_id      0
cycle           0
time            0
voltage         0
current         0
temperature     0
current_load    0
voltage_load    0
capacity        0
dtype: int64

=== DUPLICATE RECORDS ===
Duplicate rows: 0

=== DATA TYPES ===
battery_id       object
cycle             int64
time            float64
voltage         float64
current         float64
temperature     float64
current_load    float64
voltage_load    float64
capacity        float64
dtype: object

=== NUMERICAL SUMMARY ===


,cycle,time,voltage,current,temperature,current_load,voltage_load,capacity
count,185721.000000,185721.000000,185721.000000,185721.000000,185721.000000,185721.000000,185721.000000,185721.000000
mean,82.838758,1546.379935,3.497219,-1.832569,32.378997,1.465434,2.366494,1.574863
std,45.692247,906.958628,0.251691,0.561405,4.027737,1.226874,0.751377,0.190633
min,1.000000,0.000000,1.737030,-2.029098,22.350256,-2.000000,0.000000,1.153818
25%,45.000000,764.797000,3.377653,-2.011418,29.570621,1.998200,2.410000,1.426025
50%,81.000000,1537.031000,3.500859,-2.009015,32.355737,1.998800,2.558000,1.559634
75%,120.000000,2307.485000,3.655751,-1.989974,35.420677,1.999000,2.718000,1.741850
max,168.000000,3690.234000,4.233325,0.014306,42.332522,2.000000,4.249000,2.035338


In [7]:
print("=== INVALID VALUE CHECK ===")

checks = {
    "Invalid cycle numbers": (df["cycle"] <= 0).sum(),
    "Negative time values": (df["time"] < 0).sum(),
    "Non-positive voltage values": (df["voltage"] <= 0).sum(),
    "Non-positive capacity values": (df["capacity"] <= 0).sum(),
}

for check, count in checks.items():
    print(f"{check}: {count}")

print("\n=== MEASUREMENT RANGES ===")

for column in [
    "time",
    "voltage",
    "current",
    "temperature",
    "current_load",
    "voltage_load",
    "capacity"
]:
    print(
        f"{column:16s} "
        f"min={df[column].min():.4f}, "
        f"max={df[column].max():.4f}"
    )

=== INVALID VALUE CHECK ===
Invalid cycle numbers: 0
Negative time values: 0
Non-positive voltage values: 0
Non-positive capacity values: 0

=== MEASUREMENT RANGES ===
time             min=0.0000, max=3690.2340
voltage          min=1.7370, max=4.2333
current          min=-2.0291, max=0.0143
temperature      min=22.3503, max=42.3325
current_load     min=-2.0000, max=2.0000
voltage_load     min=0.0000, max=4.2490
capacity         min=1.1538, max=2.0353


In [8]:
print("=== CYCLE STRUCTURE VALIDATION ===")

for battery_id in df["battery_id"].unique():

    battery_df = df[df["battery_id"] == battery_id]

    cycles = battery_df["cycle"].unique()

    print(f"\n{battery_id}")
    print("-" * 30)

    print("Number of cycles:", len(cycles))
    print("First cycle:", cycles.min())
    print("Last cycle:", cycles.max())

    # Number of measurements per cycle
    samples_per_cycle = battery_df.groupby("cycle").size()

    print("Minimum samples/cycle:", samples_per_cycle.min())
    print("Maximum samples/cycle:", samples_per_cycle.max())
    print("Average samples/cycle:", round(samples_per_cycle.mean(), 2))

    # Check that each cycle has only one capacity value
    capacity_counts = battery_df.groupby("cycle")["capacity"].nunique()

    print(
        "Cycles with multiple capacity values:",
        (capacity_counts > 1).sum()
    )

=== CYCLE STRUCTURE VALIDATION ===

B0005
------------------------------
Number of cycles: 168
First cycle: 1
Last cycle: 168
Minimum samples/cycle: 179
Maximum samples/cycle: 371
Average samples/cycle: 299.32
Cycles with multiple capacity values: 0

B0006
------------------------------
Number of cycles: 168
First cycle: 1
Last cycle: 168
Minimum samples/cycle: 179
Maximum samples/cycle: 371
Average samples/cycle: 299.32
Cycles with multiple capacity values: 0

B0007
------------------------------
Number of cycles: 168
First cycle: 1
Last cycle: 168
Minimum samples/cycle: 179
Maximum samples/cycle: 371
Average samples/cycle: 299.32
Cycles with multiple capacity values: 0

B0018
------------------------------
Number of cycles: 132
First cycle: 1
Last cycle: 132
Minimum samples/cycle: 200
Maximum samples/cycle: 366
Average samples/cycle: 264.14
Cycles with multiple capacity values: 0


In [9]:
# Define the standardized column order
standard_columns = [
    "battery_id",
    "cycle",
    "time",
    "voltage",
    "current",
    "temperature",
    "current_load",
    "voltage_load",
    "capacity"
]

df = df[standard_columns]

# Save standardized discharge dataset
OUTPUT_FILE = PROCESSED_DIR / "battery_discharge_cycles.csv"

df.to_csv(OUTPUT_FILE, index=False)

print("Processed dataset saved successfully!")
print("\nFile:")
print(OUTPUT_FILE)

print("\nDataset shape:", df.shape)
print("File size:", round(OUTPUT_FILE.stat().st_size / (1024 * 1024), 2), "MB")

Processed dataset saved successfully!

File:
d:\VIT Academic\TY\sem 5\EDI\BatteryIQ\data\processed\battery_discharge_cycles.csv

Dataset shape: (185721, 9)
File size: 18.85 MB


In [10]:
# Reload the saved processed dataset
df_check = pd.read_csv(OUTPUT_FILE)

print("=== SAVED DATASET VALIDATION ===")

print("Shape:", df_check.shape)

print("\nColumns:")
print(df_check.columns.tolist())

print("\nMissing values:")
print(df_check.isna().sum().sum())

print("\nDuplicate rows:")
print(df_check.duplicated().sum())

print("\nBattery IDs:")
print(df_check["battery_id"].unique())

print("\nCycles per battery:")
print(df_check.groupby("battery_id")["cycle"].nunique())

print("\nCapacity range:")
print(
    f"{df_check['capacity'].min():.4f} - "
    f"{df_check['capacity'].max():.4f} Ah"
)

=== SAVED DATASET VALIDATION ===
Shape: (185721, 9)

Columns:
['battery_id', 'cycle', 'time', 'voltage', 'current', 'temperature', 'current_load', 'voltage_load', 'capacity']

Missing values:
0

Duplicate rows:
0

Battery IDs:
['B0005' 'B0006' 'B0007' 'B0018']

Cycles per battery:
battery_id
B0005    168
B0006    168
B0007    168
B0018    132
Name: cycle, dtype: int64

Capacity range:
1.1538 - 2.0353 Ah
